# Import Modules

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta
import calendar
import os

# Parameters

## 1. General Parameters

In [2]:
today = dt.datetime(2026, 10, 19, 23, 0, 0)
bom_factor = today.day / calendar.monthrange(today.year, today.month)[1]
filename="desk1.xlsx"

products = ["FUTURE", "SWAP", "PPA"]
supported_curves = [
    "DE POWER", "FR POWER", "GR POWER", "GR POWER PK", "HU POWER", 
    "TTF", 
    "EUA"] 


In [3]:
bom_factor

0.6129032258064516

## 2. Path Parameters

In [4]:
input_dir = r"C:\\Users\\g.varvounis\\Documents\\RiC\\_data"
output_dir = r"C:\\Users\\g.varvounis\\Documents\\RiC\\_other_outputs\\_pnl"

prices_dir = r"C:\\Users\\g.varvounis\\OneDrive - HELLENiQ ENERGY\\Risk Section Activity\\RiC Monitoring\\_data\\BBG"

# Import Positions File

In [5]:
df_pos = pd.read_excel(os.path.join(input_dir,filename), sheet_name="FUTURE")

In [6]:
#df_pos["START DELIVERY"].dt.day

# Import Prices

## 1. Realized Prices

In [7]:
prices_dir_specific = os.path.join(prices_dir,"SPOT")

prices_dic = {}

for curve in supported_curves:
    
    if curve[-2:] == "PK":
        curve_modified = curve.split(" ")[0] + " " + curve.split(" ")[1]
        df_temp = pd.read_excel(os.path.join(prices_dir_specific, curve_modified + " - Peak MtD.xlsx"))
        df_temp.columns = ["date", "value"]
    elif "POWER" in curve:
        df_temp = pd.read_excel(os.path.join(prices_dir_specific, curve + " - Initial Frequency.xlsx"))
        df_temp.columns = ["date", "value"]
    else:
        df_temp = pd.DataFrame({"date": 0, "value": 0}, index=[0])

    prices_dic.update({curve: df_temp})

## 2. Unrealized Prices

In [8]:
prices_dic_unrealized = {}

for curve in supported_curves:
    if ("POWER" in curve) or ("EUA" in curve):
        curve_modified = curve.split(" ")[0] + "_CURVE"
    elif ("TTF" in curve):
        curve_modified = "ICE_" + curve.split(" ")[0] + "_CURVE"

    df_temp = pd.read_excel(os.path.join(prices_dir, curve_modified + ".xlsx"))[["Tenor", "value"]]
    prices_dic_unrealized.update({curve: df_temp})
    

In [9]:
prices_dic_unrealized

{'DE POWER':           Tenor   value
 0   DE POWER W0  143.09
 1   DE POWER W1  167.26
 2   DE POWER W2  195.71
 3   DE POWER W3  188.70
 4   DE POWER W4  194.61
 5   DE POWER M0  159.96
 6   DE POWER M1  193.76
 7   DE POWER M2  188.56
 8   DE POWER M3  201.58
 9   DE POWER M4  196.32
 10  DE POWER Q1  188.35
 11  DE POWER Q2  122.75
 12  DE POWER Q3  131.68
 13  DE POWER Q4  148.85
 14  DE POWER Y1  147.75
 15  DE POWER Y2  110.64,
 'FR POWER':           Tenor   value
 0   FR POWER W0  134.71
 1   FR POWER W1  163.40
 2   FR POWER W2  176.99
 3   FR POWER W3  166.19
 4   FR POWER W4  172.50
 5   FR POWER M0  158.41
 6   FR POWER M1  170.00
 7   FR POWER M2  168.68
 8   FR POWER M3  184.02
 9   FR POWER M4  179.85
 10  FR POWER Q1  169.62
 11  FR POWER Q2   72.18
 12  FR POWER Q3   88.54
 13  FR POWER Q4  113.00
 14  FR POWER Y1  110.61
 15  FR POWER Y2   67.29,
 'GR POWER':           Tenor   value
 0   GR POWER M0  181.72
 1   GR POWER M1  210.03
 2   GR POWER M2  199.88
 3   GR POWE

## 3. Define prices per deal

In [10]:
realized_prices = []
unrealized_prices = []

for idx, deal_id in enumerate(list(df_pos["ID"])):
    # 1. Realized Price Definition
    try:
        start = pd.to_datetime(df_pos["START DELIVERY"].iloc[idx])
        end = pd.to_datetime(min(df_pos["END DELIVERY"].iloc[idx], today))
        temp_realized_price = prices_dic[df_pos["M2M"].iloc[idx]].loc[
            lambda x: (x["date"] >= start) &
                  (x["date"] <= end)]["value"].mean()
    except:
        temp_realized_price = 0

    realized_prices.append(temp_realized_price)

    # 2. Unrealized Price Definition
    try:
        temp_alias = df_pos["M2M"].iloc[idx] + " " + df_pos["Tenor"].iloc[idx]
        if "Daily" in temp_alias:
            temp_alias = df_pos["M2M"].iloc[idx] + " " + "SPOT"
            
        temp_unrealized_price = prices_dic_unrealized[df_pos["M2M"].iloc[idx]][
            prices_dic_unrealized[df_pos["M2M"].iloc[idx]]["Tenor"]==temp_alias]["value"].values[0]
    except:
        temp_unrealized_price = 0

    unrealized_prices.append(temp_unrealized_price)

In [11]:
df_pos["Realized P"] = realized_prices
df_pos["Unrealized P"] = unrealized_prices
df_pos["Realized M"]= df_pos["PRICE"] - df_pos["Realized P"]
df_pos["Unrealized M"]= df_pos["PRICE"] - df_pos["Unrealized P"]

# Calculate Exposure Parameters

## 1. Deal duration calculation

In [12]:
df_pos["Duration"] = df_pos["END DELIVERY"] - df_pos["START DELIVERY"]

## 2. Define Realized & Unrealized Quantity

In [13]:
realized_q = []
for idx, deal_id in enumerate(list(df_pos["ID"])):

    # IF baseload deal, with hourly or quarterly granularity
    if (df_pos["FREQ"].iloc[idx] == "Hourly") or (df_pos["FREQ"].iloc[idx] == "Quarterly"):
        if df_pos["Duration"].iloc[idx] <= timedelta(days=1):
            quantitiy_temp = df_pos["Q"].iloc[idx]
        else:
            quantitiy_temp = (
                min(df_pos["END DELIVERY"].iloc[idx], today) + timedelta(hours=1) - 
                df_pos["START DELIVERY"].iloc[idx]).total_seconds() / 3600 * df_pos["MW"].iloc[idx]
            quantitiy_temp = max(0, quantitiy_temp)
        if df_pos["DIR"].iloc[idx] == "Sell":
            quantitiy_temp = quantitiy_temp * (-1)
        
    # Monthly / daily deals (only unrealized P&L until maturity)
    else:
        if today < df_pos["END DELIVERY"].iloc[idx]:
            quantitiy_temp = 0
        else:
            quantitiy_temp = df_pos["Q"].iloc[idx]
        
    realized_q.append(quantitiy_temp)  

In [14]:
df_pos["Realized Q"] = realized_q
df_pos["Unrealized Q"] = df_pos["Q"] - df_pos["Realized Q"]

In [15]:
df_pos

,ID,COMM,CPY,DIR,PROD,START DELIVERY,END DELIVERY,M2M,Tenor,Product,...,dirty product,Transaction Date,Name,Realized P,Unrealized P,Realized M,Unrealized M,Duration,Realized Q,Unrealized Q
0,18803,POWER,EEX - GR,Buy,FUTURE,2026-01-01,2026-01-31 23:00:00,GR POWER,NaN,NaN,...,446400.0,2025-12-15,EEX_TFS_GREECE_BASE_ JAN26_20251215,108.673760,0.00,11.326240,120.00,30 days 23:00:00,3720.0,0.0
1,18804,POWER,EEX - GR,Buy,FUTURE,2026-02-01,2026-02-28 23:00:00,GR POWER,NaN,NaN,...,767760.0,2025-12-15,EEX_TFS_GREECE_BASE_ FEB26_20251215,78.347258,0.00,35.902742,114.25,27 days 23:00:00,6720.0,0.0
2,18877,POWER,EEX - GR,Buy,FUTURE,2026-01-01,2026-01-31 23:00:00,GR POWER,NaN,NaN,...,437286.0,2025-12-17,EEX_TFS_GREECE_BASE_ JAN26_20251217,108.673760,0.00,8.876240,117.55,30 days 23:00:00,3720.0,0.0
3,19671,POWER,EEX - GR,Buy,FUTURE,2026-02-01,2026-02-28 23:00:00,GR POWER,NaN,NaN,...,362040.0,2026-01-09,EEX_TFS_GREECE_BASE_ FEB26_20260109,78.347258,0.00,29.402742,107.75,27 days 23:00:00,3360.0,0.0
4,19672,POWER,EEX - GR,Buy,FUTURE,2026-02-01,2026-02-28 23:00:00,GR POWER,NaN,NaN,...,361200.0,2026-01-09,EEX_TFS_GREECE_BASE_ FEB26_20260109 (2),78.347258,0.00,29.152742,107.50,27 days 23:00:00,3360.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,27211,POWER,EEX - FRA,Sell,FUTURE,2026-09-28,2026-10-04 23:00:00,FR POWER,W0,Week 40,...,137970.0,2026-09-24,EEX_FRANCE_BASE_WEEK40_20260924,179.356384,134.71,-15.106384,29.54,6 days 23:00:00,-840.0,0.0
249,27266,POWER,EEX - FRA,Sell,FUTURE,2026-09-28,2026-10-04 23:00:00,FR POWER,W0,Week 40,...,83412.0,2026-09-25,EEX_FRANCE_BASE_WEEK40_20260925,179.356384,134.71,-13.856384,30.79,6 days 23:00:00,-504.0,0.0
250,27808,POWER,EEX - DE,Sell,FUTURE,2027-01-01,2027-12-31 23:00:00,DE POWER,Y1,Cal-27,...,1285705.2,2026-10-16,EEX_GERMANY_BASE_CAL27_20261016,NaN,147.75,NaN,-0.98,364 days 23:00:00,0.0,-8760.0
251,27809,POWER,EEX - DE,Sell,FUTURE,2027-01-01,2027-12-31 23:00:00,DE POWER,Y1,Cal-27,...,1285092.0,2026-10-16,EEX_GERMANY_BASE_CAL27_20261016 (2),NaN,147.75,NaN,-1.05,364 days 23:00:00,0.0,-8760.0


# Calculate P&L

In [16]:
df_pos["Realized P&L"] = df_pos["Realized M"] * df_pos["Realized Q"]
df_pos["MtM"] = df_pos["Unrealized M"] * df_pos["Unrealized Q"]

In [17]:
df_pos["Realized P&L"].sum()

np.float64(-20997573.71785074)

# Save results

In [18]:
df_pos.to_excel(os.path.join(output_dir,"PnL_" + filename + ".xlsx"))